# 11 — Loading Real Pretrained GPT-2 Weights

**Lecture goal:** load OpenAI's actual trained GPT-2 (124M) weights into our own hand-built `GPTModel`, and watch the exact same architecture — and the exact same `generate()` function from notebook 09 — go from producing word-salad (notebook 10) to fluent English.

## Why this matters

Notebook 10 proved the *mechanism* works: loss went down, the model started reproducing patterns from its training text. But `the-verdict.txt` is roughly 20,000 characters. Real GPT-2 was trained on roughly 40 **gigabytes** of internet text — millions of times more data — using far more compute than a laptop CPU over an afternoon. There's no shortcut around that gap by training longer on our tiny file; more data and more compute is genuinely what it takes.

What we *can* do cheaply: take OpenAI's actual trained weights — the literal numbers that resulted from that expensive training run — and load them directly into our own model code. If our implementation is a faithful match of the real architecture, this should "just work," and the payoff is immediate and dramatic.

## Getting the weights via Hugging Face `transformers`

GPT-2's original release distributed weights as raw TensorFlow checkpoint files, which would require a TensorFlow dependency and manual checkpoint parsing to load. Instead, we'll use the Hugging Face `transformers` library, which hosts a converted, ready-to-use copy of the same weights and downloads them for us automatically (a few hundred MB, cached locally after the first run).

We import `GPT2Model` — the base transformer *without* any task-specific head attached — since we'll be plugging its weights into our own `GPTModel`, which already defines its own output head.

In [1]:
from transformers import GPT2Model

hf_model = GPT2Model.from_pretrained("gpt2")  # the smallest GPT-2 size, ~124M parameters
hf_model.eval()

hf_state_dict = hf_model.state_dict()
print("Number of tensors in the checkpoint:", len(hf_state_dict))
for key in list(hf_state_dict.keys())[:8]:
    print(key, tuple(hf_state_dict[key].shape))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Number of tensors in the checkpoint: 148
wte.weight (50257, 768)
wpe.weight (1024, 768)
h.0.ln_1.weight (768,)
h.0.ln_1.bias (768,)
h.0.attn.c_attn.weight (768, 2304)
h.0.attn.c_attn.bias (2304,)
h.0.attn.c_proj.weight (768, 768)
h.0.attn.c_proj.bias (768,)


## Two wrinkles we need to handle

Comparing these key names and shapes against our own `GPTModel` (notebook 08) reveals two differences — not in what the model *computes*, only in how the weights happen to be *stored*.

### Wrinkle 1: `Conv1D` vs. `nn.Linear` — the weight matrix is transposed

GPT-2's original codebase used a layer type called `Conv1D` for all of its linear projections (a historical quirk, not a real convolution). Mathematically, `Conv1D` and `nn.Linear` (what our model uses) compute the exact same operation, `y = x @ W + b`, but they store the weight matrix `W` with opposite shape conventions:

- `nn.Linear` stores `W` as `(out_features, in_features)`, and internally computes `y = x @ W.T + b`.
- `Conv1D` stores `W` directly as `(in_features, out_features)`, and computes `y = x @ W + b`.

So every weight matrix we copy from the Hugging Face checkpoint into one of our `nn.Linear` layers needs to be **transposed** (`.T`) first, or the numbers will end up describing a completely different (and wrong) transformation despite the shapes matching by coincidence in some cases (square matrices!).

### Wrinkle 2: combined Q/K/V matrix

Our `MultiHeadAttention` (notebook 06) has three separate weight matrices, `W_query`, `W_key`, `W_value`. GPT-2's checkpoint instead stores one single combined matrix, `attn.c_attn.weight`, of shape `(768, 2304)` — that's `3 x 768`, all three projections concatenated together along the output dimension, computed with one larger matrix multiply for efficiency. We split it back into three equal pieces with `.chunk(3, dim=-1)`.

### Bonus: weight tying

GPT-2 also reuses (**ties**) the same weight matrix for the input token embedding *and* the final output projection — both are shape `(vocab_size, embedding_dim)`, and reusing one saves ~38 million parameters compared to learning two separate copies (this is exactly why the "124M" in GPT-2's name is smaller than the ~163M we counted for our own untied model in notebook 08). We'll copy the same tensor into both our `token_embedding` and `out_head`.

## Rebuilding our model, and a small `assign` safety helper

We reuse `GPTModel` exactly as defined in notebook 08, with the real `GPT_CONFIG_124M` configuration (note `qkv_bias=True` — unlike the small config we trained ourselves, real GPT-2's checkpoint does include a bias term on its Q/K/V projections, so our layers need to have one too, or there'll be nowhere to put those numbers).

`assign` is a tiny wrapper that double-checks the source and destination shapes match *before* copying — if we got a transpose wrong somewhere, this fails loudly and immediately instead of silently producing a broken model.

In [2]:
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        batch_size, num_tokens, d_in = x.shape
        queries = self.W_query(x).view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = self.W_key(x).view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = self.W_value(x).view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        attention_scores = queries @ keys.transpose(2, 3)
        attention_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attention_weights = torch.softmax(attention_scores / self.head_dim ** 0.5, dim=-1)
        attention_weights = self.dropout(attention_weights)
        context_vectors = (attention_weights @ values).transpose(1, 2)
        context_vectors = context_vectors.contiguous().view(batch_size, num_tokens, self.d_out)
        return self.out_proj(context_vectors)


class LayerNorm(nn.Module):
    def __init__(self, embedding_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(embedding_dim))
        self.shift = nn.Parameter(torch.zeros(embedding_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * normalized + self.shift


class FeedForward(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim), nn.GELU(), nn.Linear(4 * embedding_dim, embedding_dim)
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attention = MultiHeadAttention(
            d_in=cfg["embedding_dim"], d_out=cfg["embedding_dim"], context_length=cfg["context_length"],
            dropout=cfg["dropout_rate"], num_heads=cfg["num_heads"], qkv_bias=cfg["qkv_bias"],
        )
        self.feed_forward = FeedForward(cfg["embedding_dim"])
        self.norm1 = LayerNorm(cfg["embedding_dim"])
        self.norm2 = LayerNorm(cfg["embedding_dim"])
        self.dropout = nn.Dropout(cfg["dropout_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.attention(x)
        x = self.dropout(x)
        x = x + shortcut
        shortcut = x
        x = self.norm2(x)
        x = self.feed_forward(x)
        x = self.dropout(x)
        x = x + shortcut
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["embedding_dim"])
        self.position_embedding = nn.Embedding(cfg["context_length"], cfg["embedding_dim"])
        self.dropout = nn.Dropout(cfg["dropout_rate"])
        self.transformer_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["num_layers"])])
        self.final_norm = LayerNorm(cfg["embedding_dim"])
        self.out_head = nn.Linear(cfg["embedding_dim"], cfg["vocab_size"], bias=False)

    def forward(self, token_ids):
        batch_size, num_tokens = token_ids.shape
        token_embeds = self.token_embedding(token_ids)
        positions = torch.arange(num_tokens, device=token_ids.device)
        pos_embeds = self.position_embedding(positions)
        x = self.dropout(token_embeds + pos_embeds)
        x = self.transformer_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)


GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "embedding_dim": 768,
    "num_heads": 12,
    "num_layers": 12,
    "dropout_rate": 0.1,
    "qkv_bias": True,
}


def assign(target_param, source_tensor, name=""):
    if target_param.shape != source_tensor.shape:
        raise ValueError(f"Shape mismatch for {name}: expected {target_param.shape}, got {source_tensor.shape}")
    return nn.Parameter(source_tensor.clone().detach())

## Copying every weight across

For each of the 12 transformer blocks, we copy: the two layer norms, the split-and-transposed Q/K/V weights and biases, the transposed attention output projection, and the transposed feed-forward layers. Finally, the embeddings, the last layer norm, and the tied output head.

In [3]:
model = GPTModel(GPT_CONFIG_124M)

model.token_embedding.weight = assign(model.token_embedding.weight, hf_state_dict["wte.weight"], "wte")
model.position_embedding.weight = assign(model.position_embedding.weight, hf_state_dict["wpe.weight"], "wpe")

for b in range(GPT_CONFIG_124M["num_layers"]):
    prefix = f"h.{b}."
    block = model.transformer_blocks[b]

    q_w, k_w, v_w = hf_state_dict[prefix + "attn.c_attn.weight"].chunk(3, dim=-1)
    q_b, k_b, v_b = hf_state_dict[prefix + "attn.c_attn.bias"].chunk(3, dim=-1)
    block.attention.W_query.weight = assign(block.attention.W_query.weight, q_w.T, "W_query.weight")
    block.attention.W_key.weight = assign(block.attention.W_key.weight, k_w.T, "W_key.weight")
    block.attention.W_value.weight = assign(block.attention.W_value.weight, v_w.T, "W_value.weight")
    block.attention.W_query.bias = assign(block.attention.W_query.bias, q_b, "W_query.bias")
    block.attention.W_key.bias = assign(block.attention.W_key.bias, k_b, "W_key.bias")
    block.attention.W_value.bias = assign(block.attention.W_value.bias, v_b, "W_value.bias")

    block.attention.out_proj.weight = assign(
        block.attention.out_proj.weight, hf_state_dict[prefix + "attn.c_proj.weight"].T, "out_proj.weight"
    )
    block.attention.out_proj.bias = assign(
        block.attention.out_proj.bias, hf_state_dict[prefix + "attn.c_proj.bias"], "out_proj.bias"
    )

    block.feed_forward.layers[0].weight = assign(
        block.feed_forward.layers[0].weight, hf_state_dict[prefix + "mlp.c_fc.weight"].T, "ff.0.weight"
    )
    block.feed_forward.layers[0].bias = assign(
        block.feed_forward.layers[0].bias, hf_state_dict[prefix + "mlp.c_fc.bias"], "ff.0.bias"
    )
    block.feed_forward.layers[2].weight = assign(
        block.feed_forward.layers[2].weight, hf_state_dict[prefix + "mlp.c_proj.weight"].T, "ff.2.weight"
    )
    block.feed_forward.layers[2].bias = assign(
        block.feed_forward.layers[2].bias, hf_state_dict[prefix + "mlp.c_proj.bias"], "ff.2.bias"
    )

    block.norm1.scale = assign(block.norm1.scale, hf_state_dict[prefix + "ln_1.weight"], "norm1.scale")
    block.norm1.shift = assign(block.norm1.shift, hf_state_dict[prefix + "ln_1.bias"], "norm1.shift")
    block.norm2.scale = assign(block.norm2.scale, hf_state_dict[prefix + "ln_2.weight"], "norm2.scale")
    block.norm2.shift = assign(block.norm2.shift, hf_state_dict[prefix + "ln_2.bias"], "norm2.shift")

model.final_norm.scale = assign(model.final_norm.scale, hf_state_dict["ln_f.weight"], "final_norm.scale")
model.final_norm.shift = assign(model.final_norm.shift, hf_state_dict["ln_f.bias"], "final_norm.shift")
model.out_head.weight = assign(model.out_head.weight, hf_state_dict["wte.weight"], "out_head.weight (tied)")

print("All weights copied successfully — every shape matched.")

All weights copied successfully — every shape matched.


No shape-mismatch errors means every single one of these copies lined up correctly — strong evidence that our from-scratch `GPTModel` really does implement the same computation as the real thing, weight for weight.

## The payoff: generating text

Same `generate` function from notebook 09, completely unchanged. The only thing that's different from notebook 10's disappointing output is *which numbers* are sitting inside the model's weight matrices.

In [4]:
import tiktoken


def generate(model, token_ids, max_new_tokens, context_size, temperature=1.0, top_k=None):
    for _ in range(max_new_tokens):
        idx_condition = token_ids[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_condition)
        last_logits = logits[:, -1, :]
        if top_k is not None:
            top_k_values, _ = torch.topk(last_logits, top_k)
            min_value = top_k_values[:, -1]
            last_logits = torch.where(last_logits < min_value.unsqueeze(-1), torch.tensor(-torch.inf), last_logits)
        if temperature > 0.0:
            probabilities = torch.softmax(last_logits / temperature, dim=-1)
            next_token_id = torch.multinomial(probabilities, num_samples=1)
        else:
            next_token_id = torch.argmax(last_logits, dim=-1, keepdim=True)
        token_ids = torch.cat([token_ids, next_token_id], dim=1)
    return token_ids


model.eval()  # disable dropout - we're generating, not training
tokenizer = tiktoken.get_encoding("gpt2")

start_text = "Every effort moves you"
encoded = torch.tensor(tokenizer.encode(start_text)).unsqueeze(0)

torch.manual_seed(123)
output_ids = generate(model, encoded, max_new_tokens=30, context_size=GPT_CONFIG_124M["context_length"], temperature=0.0)
print("Greedy decoding:")
print(repr(tokenizer.decode(output_ids.squeeze(0).tolist())))

Greedy decoding:
'Every effort moves you forward.\n\nThe first step is to understand the importance of your work.\n\nThe second step is to understand the importance of your work.'


In [5]:
torch.manual_seed(123)
output_ids = generate(
    model, encoded, max_new_tokens=30, context_size=GPT_CONFIG_124M["context_length"],
    temperature=0.7, top_k=25,
)
print("Sampled decoding (temperature=0.7, top_k=25):")
print(repr(tokenizer.decode(output_ids.squeeze(0).tolist())))

Sampled decoding (temperature=0.7, top_k=25):
'Every effort moves you toward the goal of your goal, but only if you are prepared to work at it.\n\nWhen you try to push yourself to achieve something,'


Fluent, grammatical English — a stark contrast to notebook 10's `'Every effort moves you, I had been a. And. Gisburn!...'`. Nothing about our model's *code* changed between the two notebooks; only the weights did. This is the clearest possible demonstration of a central fact about modern LLMs: **the architecture is comparatively simple and well-understood (everything we built in notebooks 04–08); nearly all of the "intelligence" comes from what training on an enormous amount of data bakes into the weights.**

## Recap

- We loaded OpenAI's real, pretrained GPT-2 (124M) weights from Hugging Face directly into our own hand-built `GPTModel`, requiring only careful attention to two storage conventions: `Conv1D`'s transposed weight layout, and the combined Q/K/V matrix that needed splitting into three.
- Every weight's shape matched after transposing — solid evidence our architecture is a correct, faithful reimplementation of real GPT-2.
- The *exact same* `generate()` function, run against these real weights instead of our notebook 10 experiment, produces fluent English — the dramatic difference that data and compute scale make, using literally the same code.

### What's next

We now have a genuinely capable pretrained language model. Training a model like this from scratch is enormously expensive; what's actually practical — and what most real-world LLM applications do — is **fine-tuning**: taking a pretrained model like this one and further training it, cheaply, on a small task-specific dataset. Notebook 12 fine-tunes this model into a spam/ham text classifier; notebook 13 fine-tunes it to follow instructions.